# Performance Diagnostics: Neo4j + Databricks Federation

Independent diagnostic cells to isolate where time is spent in federated queries.
Each cell is self-contained after the configuration cell and can be run independently.

**Goal:** Determine whether slowdowns are in:
- Neo4j query execution
- Spark/SafeSpark pipeline overhead
- SQL-to-Cypher translation
- Spark Connector schema inference vs data pull
- Compound `remote_query()` CROSS JOIN overhead
- Delta lakehouse scans

**Prerequisites:**
- Neo4j UC JDBC connection configured
- SafeSpark memory settings applied
- Neo4j Spark Connector installed as cluster library
- `neo4j-uc-creds` secret scope configured

---
## Configuration

**Run this cell first.** All other cells depend on these variables.

In [ ]:
# =============================================================================
# CONFIGURATION — Run this cell first
# =============================================================================
import time

SCOPE_NAME = "neo4j-uc-creds"

# Lakehouse configuration (from secrets)
LAKEHOUSE_CATALOG = dbutils.secrets.get(SCOPE_NAME, "lakehouse_catalog")
LAKEHOUSE_SCHEMA = dbutils.secrets.get(SCOPE_NAME, "lakehouse_schema")

# Neo4j credentials from Databricks Secrets
NEO4J_HOST = dbutils.secrets.get(SCOPE_NAME, "host")
NEO4J_USERNAME = dbutils.secrets.get(SCOPE_NAME, "user")
NEO4J_PASSWORD = dbutils.secrets.get(SCOPE_NAME, "password")
try:
    NEO4J_DATABASE = dbutils.secrets.get(SCOPE_NAME, "database")
except Exception:
    NEO4J_DATABASE = "neo4j"

UC_CONNECTION_NAME = dbutils.secrets.get(SCOPE_NAME, "connection_name")
NEO4J_BOLT_URI = f"neo4j+s://{NEO4J_HOST}"

# Set catalog and schema context for Delta table queries
spark.sql(f"USE CATALOG `{LAKEHOUSE_CATALOG}`")
spark.sql(f"USE SCHEMA `{LAKEHOUSE_SCHEMA}`")

print(f"Lakehouse: {LAKEHOUSE_CATALOG}.{LAKEHOUSE_SCHEMA}")
print(f"Neo4j Host: {NEO4J_HOST}")
print(f"UC Connection: {UC_CONNECTION_NAME}")

---
## Test 1: Direct Neo4j Timing (Python Driver)

Queries Neo4j directly via the Python driver, bypassing Spark/SafeSpark entirely.
This establishes the **baseline** for how fast Neo4j can answer each query.
Compare these times against Test 2 to measure Spark pipeline overhead.

In [ ]:
# =============================================================================
# TEST 1: Direct Neo4j query timing (Python driver, no Spark)
# =============================================================================
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_BOLT_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

cypher_queries = {
    "MaintenanceEvent count":   "MATCH (m:MaintenanceEvent) RETURN COUNT(m) AS cnt",
    "Critical events":          "MATCH (m:MaintenanceEvent {severity: 'CRITICAL'}) RETURN COUNT(m) AS cnt",
    "Flight count":             "MATCH (f:Flight) RETURN COUNT(f) AS cnt",
    "Flight->Airport traversal": "MATCH (f:Flight)-[:DEPARTS_FROM]->(a:Airport) RETURN COUNT(*) AS cnt",
    "Aircraft count":           "MATCH (a:Aircraft) RETURN COUNT(a) AS cnt",
}

print("=" * 70)
print("DIRECT NEO4J (Python driver — no Spark, no SafeSpark)")
print("=" * 70)

direct_times = {}
with driver.session(database=NEO4J_DATABASE) as session:
    for label, cypher in cypher_queries.items():
        start = time.time()
        result = session.run(cypher).single()
        elapsed = time.time() - start
        direct_times[label] = elapsed
        print(f"  {label:30s} {elapsed:.3f}s  -> {result['cnt']:,}")

driver.close()
print(f"\nTotal: {sum(direct_times.values()):.3f}s")

---
## Test 2: remote_query() Timing (Individual Calls)

Runs the **same queries** as Test 1 but through `remote_query()` via UC JDBC.
Each query is timed individually. Compare against Test 1 to see the per-query
overhead added by the Spark -> SafeSpark -> JDBC -> SQL-to-Cypher pipeline.

In [ ]:
# =============================================================================
# TEST 2: remote_query() individual query timing
# =============================================================================
sql_queries = {
    "MaintenanceEvent count":   "SELECT COUNT(*) AS cnt FROM MaintenanceEvent",
    "Critical events":          "SELECT COUNT(*) AS cnt FROM MaintenanceEvent WHERE severity = 'CRITICAL'",
    "Flight count":             "SELECT COUNT(*) AS cnt FROM Flight",
    "Flight->Airport traversal": "SELECT COUNT(*) AS cnt FROM Flight f NATURAL JOIN DEPARTS_FROM r NATURAL JOIN Airport a",
    "Aircraft count":           "SELECT COUNT(*) AS cnt FROM Aircraft",
}

print("=" * 70)
print("REMOTE_QUERY() — Individual calls via UC JDBC")
print("=" * 70)

rq_times = {}
for label, sql in sql_queries.items():
    escaped = sql.replace("'", "''")
    start = time.time()
    result = spark.sql(f"""
        SELECT * FROM remote_query('{UC_CONNECTION_NAME}', query => '{escaped}')
    """).collect()
    elapsed = time.time() - start
    rq_times[label] = elapsed
    print(f"  {label:30s} {elapsed:.3f}s  -> {result[0]['cnt']:,}")

print(f"\nTotal: {sum(rq_times.values()):.3f}s")

# Compare against direct times if Test 1 was run
try:
    print(f"\n{'COMPARISON':=^70}")
    print(f"  {'Query':30s} {'Direct':>8s} {'remote_q':>8s} {'Overhead':>8s} {'Ratio':>6s}")
    print(f"  {'-'*30} {'-'*8} {'-'*8} {'-'*8} {'-'*6}")
    for label in sql_queries:
        d = direct_times.get(label, 0)
        r = rq_times.get(label, 0)
        overhead = r - d
        ratio = r / d if d > 0 else 0
        print(f"  {label:30s} {d:7.3f}s {r:7.3f}s {overhead:+7.3f}s {ratio:5.1f}x")
except NameError:
    print("\n(Run Test 1 first to see comparison)")

---
## Test 3: Delta Lakehouse Query Timing

Times queries against the local Delta lakehouse tables to establish a baseline
for the Spark-side cost. If these are slow, the bottleneck is in the Delta scan
or Spark processing, not the Neo4j federation.

In [ ]:
# =============================================================================
# TEST 3: Delta lakehouse query timing
# =============================================================================
delta_queries = {
    "sensor_readings count":  "SELECT COUNT(*) AS cnt FROM sensor_readings",
    "aircraft count":         "SELECT COUNT(*) AS cnt FROM aircraft",
    "sensors count":          "SELECT COUNT(*) AS cnt FROM sensors",
    "systems count":          "SELECT COUNT(*) AS cnt FROM systems",
    "sensor avg (full scan)": """
        SELECT
            ROUND(AVG(CASE WHEN sen.type = 'EGT' THEN r.value END), 1) AS avg_egt,
            ROUND(AVG(CASE WHEN sen.type = 'Vibration' THEN r.value END), 4) AS avg_vibration,
            COUNT(*) AS total_readings
        FROM sensor_readings r
        JOIN sensors sen ON r.sensor_id = sen.`:ID(Sensor)`
    """,
    "sensor health per aircraft": """
        SELECT
            sys.aircraft_id,
            ROUND(AVG(CASE WHEN sen.type = 'EGT' THEN r.value END), 1) AS avg_egt,
            ROUND(AVG(CASE WHEN sen.type = 'Vibration' THEN r.value END), 4) AS avg_vibration
        FROM sensor_readings r
        JOIN sensors sen ON r.sensor_id = sen.`:ID(Sensor)`
        JOIN systems sys ON sen.system_id = sys.`:ID(System)`
        GROUP BY sys.aircraft_id
    """,
}

print("=" * 70)
print("DELTA LAKEHOUSE — Local Spark SQL queries")
print("=" * 70)

delta_times = {}
for label, sql in delta_queries.items():
    start = time.time()
    result = spark.sql(sql).collect()
    elapsed = time.time() - start
    delta_times[label] = elapsed
    row_count = len(result)
    if row_count == 1 and 'cnt' in result[0].asDict():
        print(f"  {label:30s} {elapsed:.3f}s  -> {result[0]['cnt']:,} rows")
    else:
        print(f"  {label:30s} {elapsed:.3f}s  -> {row_count} result rows")

print(f"\nTotal: {sum(delta_times.values()):.3f}s")

---
## Test 4: Spark Connector — Schema Inference vs Data Pull

The Neo4j Spark Connector `.load()` does schema inference (calls Neo4j to discover
property types) before pulling any data. This test times each phase separately
to see if the slowdown is in schema inference or actual data transfer.

In [ ]:
# =============================================================================
# TEST 4: Spark Connector — schema inference vs data pull
# =============================================================================
labels_to_test = ["MaintenanceEvent", "Flight", "Aircraft"]

print("=" * 70)
print("SPARK CONNECTOR — Schema Inference vs Data Pull")
print("=" * 70)
print(f"  {'Label':25s} {'Load (schema)':>14s} {'Count (data)':>14s} {'Total':>10s}")
print(f"  {'-'*25} {'-'*14} {'-'*14} {'-'*10}")

for label in labels_to_test:
    # Phase 1: .load() — triggers schema inference
    start = time.time()
    df = spark.read.format("org.neo4j.spark.DataSource") \
        .option("url", NEO4J_BOLT_URI) \
        .option("authentication.type", "basic") \
        .option("authentication.basic.username", NEO4J_USERNAME) \
        .option("authentication.basic.password", NEO4J_PASSWORD) \
        .option("labels", label) \
        .load()
    load_time = time.time() - start

    # Phase 2: .count() — triggers actual data pull
    start = time.time()
    count = df.count()
    count_time = time.time() - start

    total = load_time + count_time
    print(f"  {label:25s} {load_time:13.3f}s {count_time:13.3f}s {total:9.3f}s  ({count:,} rows)")

---
## Test 5: Compound vs Individual remote_query()

The fleet summary in notebook 02 CROSS JOINs four `remote_query()` calls in one
SQL statement. This test compares that compound approach vs running each individually.
If the compound version is much slower, Spark may be serializing or re-evaluating
the sub-queries.

In [ ]:
# =============================================================================
# TEST 5: Compound CROSS JOIN vs individual remote_query() calls
# =============================================================================

# --- Individual calls ---
individual_queries = [
    ("MaintenanceEvent count", "SELECT COUNT(*) AS cnt FROM MaintenanceEvent"),
    ("Critical events",        "SELECT COUNT(*) AS cnt FROM MaintenanceEvent WHERE severity = 'CRITICAL'"),
    ("Flight count",           "SELECT COUNT(*) AS cnt FROM Flight"),
    ("Flight->Airport",        "SELECT COUNT(*) AS cnt FROM Flight f NATURAL JOIN DEPARTS_FROM r NATURAL JOIN Airport a"),
]

print("=" * 70)
print("INDIVIDUAL remote_query() calls")
print("=" * 70)

individual_total = 0
for label, sql in individual_queries:
    start = time.time()
    spark.sql(f"SELECT * FROM remote_query('{UC_CONNECTION_NAME}', query => '{sql}')").collect()
    elapsed = time.time() - start
    individual_total += elapsed
    print(f"  {label:30s} {elapsed:.3f}s")
print(f"  {'TOTAL':30s} {individual_total:.3f}s")

# --- Compound CROSS JOIN (same as notebook 02 fleet summary) ---
print(f"\n{'=' * 70}")
print("COMPOUND CROSS JOIN (single spark.sql with 4 remote_query calls)")
print("=" * 70)

compound_sql = f"""
    SELECT
        maint.cnt AS total_maintenance_events,
        crit.cnt AS critical_events,
        flights.cnt AS total_flights,
        deps.cnt AS flight_airport_connections
    FROM
        remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT COUNT(*) AS cnt FROM MaintenanceEvent') AS maint
    CROSS JOIN
        remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT COUNT(*) AS cnt FROM MaintenanceEvent WHERE severity = ''CRITICAL''') AS crit
    CROSS JOIN
        remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT COUNT(*) AS cnt FROM Flight') AS flights
    CROSS JOIN
        remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT COUNT(*) AS cnt FROM Flight f NATURAL JOIN DEPARTS_FROM r NATURAL JOIN Airport a') AS deps
"""

start = time.time()
spark.sql(compound_sql).collect()
compound_time = time.time() - start
print(f"  Compound CROSS JOIN:         {compound_time:.3f}s")

print(f"\n{'COMPARISON':=^70}")
print(f"  Individual total:  {individual_total:.3f}s")
print(f"  Compound total:    {compound_time:.3f}s")
diff = compound_time - individual_total
print(f"  Difference:        {diff:+.3f}s ({compound_time/individual_total:.1f}x)")

---
## Test 6: Spark Explain Plans

Shows what Spark is doing under the hood for a `remote_query()` call.
Look for unexpected stages, schema inference round-trips, or shuffle operations.

In [ ]:
# =============================================================================
# TEST 6: Spark explain plans for remote_query()
# =============================================================================

print("=" * 70)
print("EXPLAIN — Simple remote_query()")
print("=" * 70)

spark.sql(f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT COUNT(*) AS cnt FROM MaintenanceEvent')
""").explain("extended")

print(f"\n{'=' * 70}")
print("EXPLAIN — Compound CROSS JOIN with 4 remote_query() calls")
print("=" * 70)

spark.sql(f"""
    SELECT
        maint.cnt AS total_maintenance_events,
        crit.cnt AS critical_events
    FROM
        remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT COUNT(*) AS cnt FROM MaintenanceEvent') AS maint
    CROSS JOIN
        remote_query('{UC_CONNECTION_NAME}',
            query => 'SELECT COUNT(*) AS cnt FROM Flight') AS crit
""").explain("extended")

---
## Test 7: Federated Query Breakdown

Takes the full federated query from notebook 02 (sensor health + maintenance
correlation) and breaks it into individually timed steps:
1. Load Neo4j data via Spark Connector
2. Create temp view
3. Run Delta-only portion
4. Run the final JOIN

This pinpoints which step dominates the total time.

In [ ]:
# =============================================================================
# TEST 7: Federated query step-by-step breakdown
# =============================================================================
print("=" * 70)
print("FEDERATED QUERY BREAKDOWN — Sensor + Maintenance Correlation")
print("=" * 70)
timings = {}

# Step 1: Load maintenance events from Neo4j via Spark Connector
start = time.time()
neo4j_maint = spark.read.format("org.neo4j.spark.DataSource") \
    .option("url", NEO4J_BOLT_URI) \
    .option("authentication.type", "basic") \
    .option("authentication.basic.username", NEO4J_USERNAME) \
    .option("authentication.basic.password", NEO4J_PASSWORD) \
    .option("labels", "MaintenanceEvent") \
    .load()
timings["1. Spark Connector .load()"] = time.time() - start

# Step 2: Create temp view + materialize count
start = time.time()
neo4j_maint.createOrReplaceTempView("neo4j_maintenance_perf")
maint_count = neo4j_maint.count()
timings["2. Temp view + count"] = time.time() - start

# Step 3: Delta-only sensor health aggregation
start = time.time()
spark.sql("""
    SELECT
        sys.aircraft_id,
        ROUND(AVG(CASE WHEN sen.type = 'EGT' THEN r.value END), 1) AS avg_egt,
        ROUND(AVG(CASE WHEN sen.type = 'Vibration' THEN r.value END), 4) AS avg_vibration
    FROM sensor_readings r
    JOIN sensors sen ON r.sensor_id = sen.`:ID(Sensor)`
    JOIN systems sys ON sen.system_id = sys.`:ID(System)`
    GROUP BY sys.aircraft_id
""").createOrReplaceTempView("sensor_health_perf")
spark.sql("SELECT COUNT(*) FROM sensor_health_perf").collect()
timings["3. Delta sensor aggregation"] = time.time() - start

# Step 4: Neo4j maintenance aggregation (from temp view)
start = time.time()
spark.sql("""
    SELECT
        aircraft_id,
        COUNT(*) AS total_events,
        SUM(CASE WHEN severity = 'CRITICAL' THEN 1 ELSE 0 END) AS critical
    FROM neo4j_maintenance_perf
    GROUP BY aircraft_id
""").createOrReplaceTempView("maint_summary_perf")
spark.sql("SELECT COUNT(*) FROM maint_summary_perf").collect()
timings["4. Maintenance aggregation"] = time.time() - start

# Step 5: Final JOIN
start = time.time()
result = spark.sql("""
    SELECT
        a.tail_number,
        a.model,
        COALESCE(m.total_events, 0) AS maint_events,
        COALESCE(m.critical, 0) AS critical,
        s.avg_egt,
        s.avg_vibration
    FROM (SELECT `:ID(Aircraft)` AS aircraft_id, tail_number, model FROM aircraft) a
    LEFT JOIN maint_summary_perf m ON a.aircraft_id = m.aircraft_id
    LEFT JOIN sensor_health_perf s ON a.aircraft_id = s.aircraft_id
    ORDER BY maint_events DESC
""").collect()
timings["5. Final JOIN + collect"] = time.time() - start

# Results
total = sum(timings.values())
print(f"\n  {'Step':35s} {'Time':>8s} {'% of Total':>10s}")
print(f"  {'-'*35} {'-'*8} {'-'*10}")
for step, t in timings.items():
    pct = (t / total) * 100
    bar = '#' * int(pct / 2)
    print(f"  {step:35s} {t:7.3f}s {pct:8.1f}%  {bar}")
print(f"  {'-'*35} {'-'*8}")
print(f"  {'TOTAL':35s} {total:7.3f}s")
print(f"\n  Maintenance events loaded: {maint_count:,}")
print(f"  Final result rows: {len(result)}")

---
## Test 8: Warm-up Effect — Repeated Query Timing

Runs the same `remote_query()` call multiple times to see if there's a warm-up
effect within the SafeSpark sandbox (connection pooling, translation caching, JIT).
If run 1 is much slower than runs 2-5, the overhead is in connection/sandbox init.
If all runs are equally slow, the overhead is per-query.

In [ ]:
# =============================================================================
# TEST 8: Repeated query timing — warm-up effect
# =============================================================================
NUM_RUNS = 5
test_sql = f"""
    SELECT * FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT COUNT(*) AS cnt FROM MaintenanceEvent')
"""

print("=" * 70)
print(f"REPEATED QUERY — {NUM_RUNS} runs of the same remote_query()")
print("=" * 70)

run_times = []
for i in range(NUM_RUNS):
    start = time.time()
    spark.sql(test_sql).collect()
    elapsed = time.time() - start
    run_times.append(elapsed)
    marker = " <-- first (cold)" if i == 0 else ""
    print(f"  Run {i+1}: {elapsed:.3f}s{marker}")

print(f"\n  First run:        {run_times[0]:.3f}s")
print(f"  Avg (runs 2-{NUM_RUNS}):   {sum(run_times[1:]) / len(run_times[1:]):.3f}s")
print(f"  Min:              {min(run_times):.3f}s")
print(f"  Max:              {max(run_times):.3f}s")
warm_penalty = run_times[0] - (sum(run_times[1:]) / len(run_times[1:]))
print(f"  Warm-up penalty:  {warm_penalty:+.3f}s")

---
## Test 9: Summary Dashboard

Aggregates results from previous tests into a single comparison table.
Run all prior tests first, then run this cell for the overview.

In [ ]:
# =============================================================================
# TEST 9: Summary dashboard — aggregates results from prior tests
# =============================================================================
print("=" * 70)
print("PERFORMANCE SUMMARY")
print("=" * 70)

# Direct vs remote_query comparison
try:
    print(f"\n--- Neo4j: Direct Python Driver vs remote_query() ---")
    print(f"  {'Query':30s} {'Direct':>8s} {'remote_q':>8s} {'Overhead':>8s} {'Ratio':>6s}")
    print(f"  {'-'*30} {'-'*8} {'-'*8} {'-'*8} {'-'*6}")
    for label in direct_times:
        d = direct_times[label]
        r = rq_times.get(label, 0)
        if r > 0:
            overhead = r - d
            ratio = r / d if d > 0 else 0
            print(f"  {label:30s} {d:7.3f}s {r:7.3f}s {overhead:+7.3f}s {ratio:5.1f}x")
except NameError:
    print("\n  [Skipped] Run Tests 1 and 2 for Direct vs remote_query comparison")

# Delta timing
try:
    print(f"\n--- Delta Lakehouse Baselines ---")
    for label, t in delta_times.items():
        print(f"  {label:30s} {t:7.3f}s")
except NameError:
    print("\n  [Skipped] Run Test 3 for Delta baselines")

# Compound vs individual
try:
    print(f"\n--- Compound vs Individual remote_query() ---")
    print(f"  Individual total:  {individual_total:.3f}s")
    print(f"  Compound total:    {compound_time:.3f}s")
    print(f"  Compound overhead: {compound_time - individual_total:+.3f}s")
except NameError:
    print("\n  [Skipped] Run Test 5 for compound vs individual comparison")

# Warm-up
try:
    print(f"\n--- Warm-up Effect ---")
    print(f"  First run:       {run_times[0]:.3f}s")
    print(f"  Avg subsequent:  {sum(run_times[1:]) / len(run_times[1:]):.3f}s")
    print(f"  Warm-up penalty: {warm_penalty:+.3f}s")
except NameError:
    print("\n  [Skipped] Run Test 8 for warm-up analysis")

# Federated breakdown
try:
    print(f"\n--- Federated Query Breakdown ---")
    total = sum(timings.values())
    for step, t in timings.items():
        pct = (t / total) * 100
        print(f"  {step:35s} {t:7.3f}s ({pct:.0f}%)")
except NameError:
    print("\n  [Skipped] Run Test 7 for federated breakdown")

print(f"\n{'=' * 70}")
print("Interpretation guide:")
print("  - Direct ~= remote_query()  -> overhead is minimal, look at Delta/JOINs")
print("  - remote_query() >> Direct   -> overhead is in Spark/SafeSpark/JDBC pipeline")
print("  - Compound >> Individual sum -> Spark adds per-subquery overhead in CROSS JOIN")
print("  - Run 1 >> Runs 2-5          -> one-time sandbox/connection init cost")
print("  - All runs equally slow      -> per-query overhead (translation? connection?)")